In [1]:
import json
import csv
import os
import random
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

BASE = "/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset"
OUT_DIR = "/kaggle/working/kg"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# GLOBAL ENTITY MAPPER (FIXED)
# =========================================================
class GlobalEntityMapper:
    def __init__(self):
        self.map = {}
        # Lưu lại type để biết ID này là User hay Business
        self.entity_types = {}

    def get(self, key, entity_type):
        if key not in self.map:
            new_id = len(self.map)
            self.map[key] = new_id
            self.entity_types[new_id] = entity_type
        return self.map[key]

    def __len__(self):
        return len(self.map)

mapper = GlobalEntityMapper()

# Define Relations
RELATIONS = {
    "REVIEWED": 0,
    "WROTE_TIP": 1,
    "BELONGS_TO": 2,
    "IN_CATEGORY": 3,
    "LOCATED_IN": 4,
    "IN_STATE": 5,
    "FRIENDS_WITH": 6
}

# =========================================================
# STEP 1 — USERS + SOCIAL GROUP
# =========================================================
def build_users(user_path, n_clusters=5):
    users = []
    with open(user_path, "r", encoding="utf-8") as f:
        for line in f:
            u = json.loads(line)
            uid = u["user_id"]
            elite = u.get("elite", "")
            elite_years = len([e for e in elite.split(",") if e.strip()]) if isinstance(elite, str) else 0
            friends = u.get("friends", "")
            friend_count = len([x for x in friends.split(",") if x.strip() and x != "None"])

            users.append({
                "user_id": uid,
                "review_count": u.get("review_count", 0),
                "elite_years": elite_years,
                "friend_count": friend_count
            })

    df = pd.DataFrame(users)
    X = StandardScaler().fit_transform(df[["review_count", "elite_years", "friend_count"]])
    df["group_id"] = KMeans(n_clusters=n_clusters, random_state=42, n_init=10).fit_predict(X)

    # Đăng ký các User và Group vào Global Mapper
    for uid in df["user_id"]:
        mapper.get(f"U_{uid}", "User")  # Thêm prefix U_ để tránh trùng lặp string ngẫu nhiên
        
    group_names = [f"SocialGroup_{i}" for i in range(n_clusters)]
    for gname in group_names:
        mapper.get(gname, "SocialGroup")

    group_id_to_name = {i: group_names[i] for i in range(n_clusters)}
    df["group_name"] = df["group_id"].map(group_id_to_name)

    return df

# =========================================================
# STEP 2 — BUSINESSES
# =========================================================
def load_businesses(path):
    biz = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            b = json.loads(line)
            bid = b["business_id"]
            cats = [c.strip() for c in (b.get("categories") or "").split(",") if c.strip()]
            city = (b.get("city") or "Unknown").strip()
            state = (b.get("state") or "Unknown").strip()

            biz[bid] = {"categories": cats, "city": city, "state": state}

            # Đăng ký Entity vào Global Mapper
            mapper.get(f"B_{bid}", "Business")
            for c in cats:
                mapper.get(f"Cat_{c}", "Category")
            mapper.get(f"City_{city}", "City")
            mapper.get(f"State_{state}", "State")

    return biz

# =========================================================
# STEP 3 — BUILD TRIPLES (FIXED)
# =========================================================
def build_triples(review_path, tip_path, user_path, user_df, biz):
    triples = []
    stats = defaultdict(int)

    # Dùng set để check Entity tồn tại nhanh hơn
    valid_users = set([f"U_{u}" for u in user_df["user_id"]])
    valid_biz = set([f"B_{b}" for b in biz.keys()])

    # 1. REVIEWED (Fix: Dedup)
    print("Building REVIEWED...")
    seen_reviews = set()
    with open(review_path, "r", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            u_key, b_key = f"U_{r['user_id']}", f"B_{r['business_id']}"
            if u_key in valid_users and b_key in valid_biz:
                if (u_key, b_key) not in seen_reviews:
                    seen_reviews.add((u_key, b_key))
                    triples.append((mapper.get(u_key, ""), RELATIONS["REVIEWED"], mapper.get(b_key, "")))
                    stats["REVIEWED"] += 1

    # 2. WROTE_TIP
    print("Building WROTE_TIP...")
    seen_tips = set()
    with open(tip_path, "r", encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)
            u_key, b_key = f"U_{t['user_id']}", f"B_{t['business_id']}"
            if u_key in valid_users and b_key in valid_biz:
                if (u_key, b_key) not in seen_tips:
                    seen_tips.add((u_key, b_key))
                    triples.append((mapper.get(u_key, ""), RELATIONS["WROTE_TIP"], mapper.get(b_key, "")))
                    stats["WROTE_TIP"] += 1

    # 3. BELONGS_TO
    print("Building BELONGS_TO...")
    for _, row in user_df.iterrows():
        u_key = f"U_{row['user_id']}"
        g_key = row['group_name']
        triples.append((mapper.get(u_key, ""), RELATIONS["BELONGS_TO"], mapper.get(g_key, "")))
        stats["BELONGS_TO"] += 1

    # 4. IN_CATEGORY & LOCATED_IN & IN_STATE
    print("Building BUSINESS ATTRIBUTES...")
    seen_state_city = set()  # Fix lỗi duplicate IN_STATE
    for bid, info in biz.items():
        b_key = f"B_{bid}"
        city_key = f"City_{info['city']}"
        state_key = f"State_{info['state']}"

        # Category
        for c in info["categories"]:
            cat_key = f"Cat_{c}"
            triples.append((mapper.get(b_key, ""), RELATIONS["IN_CATEGORY"], mapper.get(cat_key, "")))
            stats["IN_CATEGORY"] += 1

        # Located In
        triples.append((mapper.get(b_key, ""), RELATIONS["LOCATED_IN"], mapper.get(city_key, "")))
        stats["LOCATED_IN"] += 1

        # In State (Chỉ thêm 1 lần cho mỗi cặp City-State)
        if (city_key, state_key) not in seen_state_city:
            seen_state_city.add((city_key, state_key))
            triples.append((mapper.get(city_key, ""), RELATIONS["IN_STATE"], mapper.get(state_key, "")))
            stats["IN_STATE"] += 1

    # 5. FRIENDS_WITH (10% sampling)
    print("Building FRIENDS_WITH...")
    random.seed(42)
    with open(user_path, "r", encoding="utf-8") as f:
        for line in f:
            u = json.loads(line)
            uid = u["user_id"]
            u_key = f"U_{uid}"
            
            if u_key not in valid_users:
                continue

            friends = u.get("friends", "")
            if not friends or friends == "None":
                continue

            friend_list = [x.strip() for x in friends.split(",") if x.strip()]
            for fid in friend_list:
                f_key = f"U_{fid}"
                if f_key in valid_users and uid < fid:
                    if random.random() <= 0.1:  # 10% sample
                        triples.append((mapper.get(u_key, ""), RELATIONS["FRIENDS_WITH"], mapper.get(f_key, "")))
                        stats["FRIENDS_WITH"] += 1

    return triples, stats

# =========================================================
# STEP 4 — SAVE
# =========================================================
def save(triples, stats):
    # Save Triples
    arr = np.array(triples, dtype=np.int32)
    np.save(f"{OUT_DIR}/kg_triples.npy", arr)

    with open(f"{OUT_DIR}/kg_triples.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["head", "relation", "tail"])
        writer.writerows(triples)

    with open(f"{OUT_DIR}/kg_stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    # Save Mapping (Quan trọng để sau này lookup ID)
    with open(f"{OUT_DIR}/entity_mapping.json", "w") as f:
        json.dump(mapper.map, f)
        
    with open(f"{OUT_DIR}/entity_types.json", "w") as f:
        json.dump(mapper.entity_types, f)

    # Save Relation Map
    with open(f"{OUT_DIR}/relation_mapping.json", "w") as f:
        json.dump(RELATIONS, f)

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    USER_PATH = f"{BASE}/yelp_academic_dataset_user.json"
    BIZ_PATH = f"{BASE}/yelp_academic_dataset_business.json"
    REVIEW_PATH = f"{BASE}/yelp_academic_dataset_review.json"
    TIP_PATH = f"{BASE}/yelp_academic_dataset_tip.json"

    print("Loading users...")
    user_df = build_users(USER_PATH)

    print("Loading businesses...")
    biz = load_businesses(BIZ_PATH)

    print(f"Total Entities in Global Space: {len(mapper)}")

    print("Building triples...")
    triples, stats = build_triples(REVIEW_PATH, TIP_PATH, USER_PATH, user_df, biz)

    print("Saving...")
    save(triples, stats)

    print("DONE ✔ KG built successfully!")


Loading users...
Loading businesses...
Total Entities in Global Space: 2140967
Building triples...
Building REVIEWED...
Building WROTE_TIP...
Building BELONGS_TO...
Building BUSINESS ATTRIBUTES...
Building FRIENDS_WITH...
Saving...
DONE ✔ KG built successfully!
